# 2019 — BERT Phishing Classifier — Eval Walkthrough

Loads the held-out eval results and walks through:

1. Headline metrics (acc / precision / recall / F1)
2. ROC + confusion matrix
3. Training curves
4. Error analysis: where does the model misfire?


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100


## 1. Headline metrics

In [ ]:
with open('results/eval_summary.json') as f: m = json.load(f)
print(json.dumps(m, indent=2))

{
  "model": "bert-base-uncased fine-tuned",
  "n_test": 3900,
  "accuracy": 0.9726,
  "precision": {"benign": 0.977, "phishing": 0.969},
  "recall": {"benign": 0.967, "phishing": 0.978},
  "f1": {"benign": 0.972, "phishing": 0.973},
  "auc": 0.987,
  ...
}

BERT-base out of the box hits 0.97+ accuracy on a balanced phishing/benign corpus after 3 epochs. Beats the 2018-era LSTM baseline (~0.92 AUC) by ~6pp on AUC.

## 2. ROC + confusion matrix

In [ ]:
from IPython.display import Image
Image('figures/roc.png')

<Figure (see figures/roc.png)>

In [ ]:
Image('figures/confusion_matrix.png')

<Figure (see figures/confusion_matrix.png)>

63 false positives + 44 false negatives across 3900 examples. False-positive cost matters here — flagging a real customer email as phishing is high-friction. Calibrate the decision threshold for your deployment context.

## 3. Training curves

In [ ]:
Image('figures/training_curves.png')

<Figure (see figures/training_curves.png)>

## 4. Error analysis (sample)

In [ ]:
# Synthetic example errors for illustration
errors = pd.DataFrame([
    {'text_short': 'Reminder: invoice #4421 due...', 'true': 0, 'pred': 1, 'prob': 0.62},
    {'text_short': 'Click <URL> to verify your...', 'true': 1, 'pred': 0, 'prob': 0.41},
    {'text_short': 'Q3 financial summary attached', 'true': 0, 'pred': 1, 'prob': 0.55},
])
errors

                          text_short  true  pred  prob
0    Reminder: invoice #4421 due...     0     1  0.62
1    Click <URL> to verify your...      1     0  0.41
2    Q3 financial summary attached     0     1  0.55


Common error patterns:
- **Transactional benign** flagged as phishing — invoice reminders share lexical features with phishing.
- **Stripped phishing** missed — once URLs and signatures are removed, body text alone may not carry signal.
- **Mitigation:** keep URL features (length, IP-in-URL, protocol-mix) as auxiliary inputs alongside the text.